# Célula 1: Importação de Bibliotecas e Configuração do Ambiente

In [1]:
import os
from urllib.parse import quote_plus
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# Carrega as variáveis de ambiente do ficheiro .env
load_dotenv()
print("✅ Mini Módulo 1: Bibliotecas e variáveis de ambiente configuradas!")

✅ Mini Módulo 1: Bibliotecas e variáveis de ambiente configuradas!


# Célula 2: Conexão com o Banco de Dados (Engine) 

In [2]:
def create_db_engine():
    user = quote_plus(os.getenv("DB_USER", "postgres"))
    password = quote_plus(os.getenv("DB_PASS", "password"))
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    dbname = os.getenv("DB_NAME", "data_lake")

    connection_str = (
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    )
    return create_engine(connection_str)


# Inicializa o motor de conexão global do notebook
engine = create_db_engine()
print("🔌 Mini Módulo 2: Conexão com o banco de dados estabelecida!")

🔌 Mini Módulo 2: Conexão com o banco de dados estabelecida!


# Célula 3: Extração e Pivotagem das Variáveis de Primeira Ordem (Silver)

In [3]:
def extrair_dados_silver(engine):
    print("📥 A extrair e a agregar dados das tabelas Silver...")

    # 1. Query do Balanço Patrimonial estruturando as variáveis V00 a V15
    query_bp = """
    SELECT 
        "CNPJ_CIA",
        "DENOM_CIA" AS "RAZAO_SOCIAL",
        "DT_FIM_EXERC_TRATADO" AS "DT_REFER",
        MAX(CASE WHEN "CD_CONTA" = '1' THEN "VL_CONTA_TRATADO" END) AS "V00",
        MAX(CASE WHEN "CD_CONTA" = '1.01' THEN "VL_CONTA_TRATADO" END) AS "V01",
        MAX(CASE WHEN "CD_CONTA" = '1.01.01' THEN "VL_CONTA_TRATADO" END) AS "V02",
        MAX(CASE WHEN "CD_CONTA" = '1.01.02' THEN COALESCE("VL_CONTA_TRATADO", 0) END) AS "V03",
        MAX(CASE WHEN "CD_CONTA" = '1.01.03.01' THEN COALESCE("VL_CONTA_TRATADO", 0) END) AS "V04",
        MAX(CASE WHEN "CD_CONTA" = '1.01.04' THEN COALESCE("VL_CONTA_TRATADO", 0) END) AS "V05",
        MAX(CASE WHEN "CD_CONTA" = '1.02.01' THEN "VL_CONTA_TRATADO" END) AS "V06",
        MAX(CASE WHEN "CD_CONTA" = '1.02.03' THEN "VL_CONTA_TRATADO" END) AS "V07",
        MAX(CASE WHEN "CD_CONTA" = '1.02.04' THEN "VL_CONTA_TRATADO" END) AS "V08",
        MAX(CASE WHEN "CD_CONTA" = '2' THEN "VL_CONTA_TRATADO" END) AS "V09",
        MAX(CASE WHEN "CD_CONTA" = '2.01' THEN "VL_CONTA_TRATADO" END) AS "V10",
        MAX(CASE WHEN "CD_CONTA" = '2.01.02' THEN "VL_CONTA_TRATADO" END) AS "V11",
        MAX(CASE WHEN "CD_CONTA" = '2.01.04' THEN COALESCE("VL_CONTA_TRATADO", 0) END) AS "V12",
        MAX(CASE WHEN "CD_CONTA" = '2.02' THEN "VL_CONTA_TRATADO" END) AS "V13",
        MAX(CASE WHEN "CD_CONTA" = '2.02.01' THEN COALESCE("VL_CONTA_TRATADO", 0) END) AS "V14",
        MAX(CASE WHEN "CD_CONTA" = '2.03' THEN "VL_CONTA_TRATADO" END) AS "V15"
    FROM layer_02_silver.n1_dfp_cia_aberta_bp
    GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_FIM_EXERC_TRATADO";
    """

    # 2. Query da DRE estruturando as variáveis V16 a V23
    query_dre = """
    SELECT 
        "CNPJ_CIA",
        "DT_FIM_EXERC_TRATADO" AS "DT_REFER",
        MAX(CASE WHEN "CD_CONTA" = '3.01' THEN "VL_CONTA_TRATADO" END) AS "V16",
        MAX(CASE WHEN "CD_CONTA" = '3.02' THEN "VL_CONTA_TRATADO" END) AS "V17",
        MAX(CASE WHEN "CD_CONTA" = '3.03' THEN "VL_CONTA_TRATADO" END) AS "V18",
        MAX(CASE WHEN "CD_CONTA" = '3.04' THEN "VL_CONTA_TRATADO" END) AS "V19",
        MAX(CASE WHEN "CD_CONTA" = '3.05' THEN "VL_CONTA_TRATADO" END) AS "V20",
        MAX(CASE WHEN "CD_CONTA" = '3.06' THEN "VL_CONTA_TRATADO" END) AS "V21",
        MAX(CASE WHEN "CD_CONTA" = '3.07' THEN "VL_CONTA_TRATADO" END) AS "V22",
        MAX(CASE WHEN "CD_CONTA" = '3.11' THEN "VL_CONTA_TRATADO" END) AS "V23"
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    GROUP BY "CNPJ_CIA", "DT_FIM_EXERC_TRATADO";
    """

    df_bp = pd.read_sql(query_bp, engine)
    df_dre = pd.read_sql(query_dre, engine)

    # Consolida os dados através de um Inner Join pelas chaves primárias de negócio
    df_base = pd.merge(df_bp, df_dre, on=["CNPJ_CIA", "DT_REFER"], how="inner")

    # Mapeamento de colunas de metadados exigidas pelo modelo físico da Gold
    df_base["SETOR"] = "Não Informado"
    df_base["TP_MERC"] = "Bovespa"

    return df_base


df_primeira_ordem = extrair_dados_silver(engine)
print(
    f"📊 Dados extraídos com sucesso! Formato atual: {df_primeira_ordem.shape}"
)

📥 A extrair e a agregar dados das tabelas Silver...
📊 Dados extraídos com sucesso! Formato atual: (2220, 29)


# Célula 4: Engenharia de Funcionalidades (Segunda Ordem e Indicadores)

In [4]:
def calcular_estruturas_financeiras(df):
    print("🧮 A calcular indicadores e variáveis de segunda ordem...")

    # Garante que dados nulos não quebrem as operações aritméticas do algoritmo
    df = df.fillna(0)

    # --- VARIÁVEIS DE SEGUNDA ORDEM ---
    df["CGL"] = df["V01"] - df["V10"]  # Capital de Giro Livre
    df["CGP"] = df["V15"] - (
        df["V06"] + df["V07"] + df["V08"]
    )  # Capital de Giro Próprio
    df["NCG"] = (
        df["V04"] + df["V05"]
    ) - df["V11"]  # Necessidade de Capital de Giro
    df["ST"] = (df["V02"] + df["V03"]) - df["V12"]  # Saldo de Tesouraria

    # --- GRUPO 1: LIQUIDEZ ---
    df["LIQUIDEZ_CORRENTE"] = np.where(df["V10"] != 0, df["V01"] / df["V10"], 0)
    df["LIQUIDEZ_SECA"] = np.where(
        df["V10"] != 0, (df["V01"] - df["V05"]) / df["V10"], 0
    )
    df["LIQUIDEZ_IMEDIATA"] = np.where(df["V10"] != 0, df["V02"] / df["V10"], 0)
    df["LIQUIDEZ_GERAL"] = np.where(
        (df["V10"] + df["V13"]) != 0,
        (df["V01"] + df["V06"]) / (df["V10"] + df["V13"]),
        0,
    )

    # --- GRUPO 2: ENDIVIDAMENTO ---
    df["ENDIVIDAMENTO_GERAL"] = np.where(
        df["V00"] != 0, (df["V10"] + df["V13"]) / df["V00"], 0
    )
    df["GRAU_ENDIVIDAMENTO"] = np.where(
        df["V15"] != 0, (df["V10"] + df["V13"]) / df["V15"], 0
    )
    df["COMPOSICAO_ENDIVIDAMENTO"] = np.where(
        (df["V10"] + df["V13"]) != 0, df["V10"] / (df["V10"] + df["V13"]), 0
    )
    df["DIVIDA_LIQUIDA"] = (df["V12"] + df["V14"]) - (df["V02"] + df["V03"])

    # --- GRUPO 3: MARGENS ---
    df["MARGEM_BRUTA"] = np.where(df["V16"] != 0, df["V18"] / df["V16"], 0)
    df["MARGEM_EBIT"] = np.where(df["V16"] != 0, df["V20"] / df["V16"], 0)
    df["MARGEM_LIQUIDA"] = np.where(df["V16"] != 0, df["V23"] / df["V16"], 0)

    # --- GRUPO 4: RENTABILIDADE ---
    df["ROA"] = np.where(df["V00"] != 0, df["V23"] / df["V00"], 0)
    df["ROE"] = np.where(df["V15"] != 0, df["V23"] / df["V15"], 0)

    # --- GRUPO 5 E 6: ATIVIDADE E CICLOS ---
    df["PMRV"] = np.where(df["V16"] != 0, (df["V04"] / df["V16"]) * 360, 0)
    df["PME"] = np.where(df["V17"] != 0, (df["V05"] / abs(df["V17"])) * 360, 0)
    df["PMPF"] = np.where(df["V17"] != 0, (df["V11"] / abs(df["V17"])) * 360, 0)

    df["CICLO_OPERACIONAL"] = df["PME"] + df["PMRV"]
    df["CICLO_FINANCEIRO"] = df["CICLO_OPERACIONAL"] - df["PMPF"]

    return df


df_gold_final = calcular_estruturas_financeiras(df_primeira_ordem)
print("✅ Mini Módulo 4: Variáveis calculadas com sucesso!")

🧮 A calcular indicadores e variáveis de segunda ordem...
✅ Mini Módulo 4: Variáveis calculadas com sucesso!


# Célula 5: Persistência na Camada Gold (mart_indicadores_financeiros)

In [5]:
def carregar_camada_gold(df, engine):
    print("🚀 A persistir os dados na tabela final da camada Gold...")

    # Garante fisicamente a existência do Schema de Data Marts
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS layer_03_gold;"))
        conn.commit()

    # Salva o DataFrame respeitando a tipagem e ordenação solicitada
    df.to_sql(
        name="mart_indicadores_financeiros",
        con=engine,
        schema="layer_03_gold",
        if_exists="replace",
        index=False,
    )
    print(
        "✨ Tabela 'layer_03_gold.mart_indicadores_financeiros' criada e povoada!"
    )


carregar_camada_gold(df_gold_final, engine)

🚀 A persistir os dados na tabela final da camada Gold...
✨ Tabela 'layer_03_gold.mart_indicadores_financeiros' criada e povoada!


# Célula 6: Validação Cruzada (Data Quality Check)

In [6]:
# Módulo de teste rápido para validar se a tabela está legível e estruturada corretamente
with engine.connect() as conn:
    validacao = pd.read_sql(
        text(
            'SELECT "CNPJ_CIA", "DT_REFER", "RAZAO_SOCIAL", "SETOR", "LIQUIDEZ_CORRENTE" FROM layer_03_gold.mart_indicadores_financeiros LIMIT 5;'
        ),
        conn,
    )

print("🔍 Amostra dos dados gravados na Gold:")
display(validacao)

🔍 Amostra dos dados gravados na Gold:


,CNPJ_CIA,DT_REFER,RAZAO_SOCIAL,SETOR,LIQUIDEZ_CORRENTE
0,00.070.698/0001-11,2010-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Não Informado,0.716558
1,00.070.698/0001-11,2011-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Não Informado,0.695209
2,00.070.698/0001-11,2012-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Não Informado,0.927259
3,00.070.698/0001-11,2013-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Não Informado,0.580409
4,00.070.698/0001-11,2014-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Não Informado,0.823033
